In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import mlx .core as mx

from storm_track_mask_processing import (
build_distance_threshold_sq_by_km ,
build_track_mask_directories ,
count_track_mask_files ,
missing_track_mask_radii ,
save_track_mask_variants ,
)

from tqdm import tqdm


In [2]:
weather_db_directory =Path ('/Users/aaronspaulding/data/weather_db_hurricane_regions/')
track_mask_radii_km =(150 ,200 ,250 ,300 ,350 )
storm_track_mask_directories =build_track_mask_directories (weather_db_directory ,track_mask_radii_km )
storm_track_mask_directory =storm_track_mask_directories [200 ]

distance_threshold_sq_by_km =build_distance_threshold_sq_by_km (track_mask_radii_km )


In [3]:
relevant_storms =gpd .read_feather ('data_cache/relevant_storm_tracks.feather')
relevant_storms =relevant_storms [::-1 ].reset_index (drop =True )

relevant_storms [['SID','NAME','datetime_min','datetime_max']]


,SID,NAME,datetime_min,datetime_max
0,2024279N21265,MILTON,2024-10-09 18:00:00+00:00,2024-10-10 15:00:00+00:00
1,2024268N17278,HELENE,2024-09-26 21:00:00+00:00,2024-09-28 18:00:00+00:00
2,2024259N31284,UNNAMED,2024-09-15 18:00:00+00:00,2024-09-17 00:00:00+00:00
3,2024253N21266,FRANCINE,2024-09-10 12:00:00+00:00,2024-09-14 00:00:00+00:00
4,2024225N14313,ERNESTO,2024-08-13 18:00:00+00:00,2024-08-14 15:00:00+00:00
...,...,...,...,...
76,2015167N27266,BILL,2015-06-16 03:00:00+00:00,2015-06-21 00:00:00+00:00
77,2015126N27281,ANA,2015-05-06 06:00:00+00:00,2015-05-12 18:00:00+00:00
78,2014285N16305,GONZALO,2014-10-13 22:45:00+00:00,2014-10-14 15:00:00+00:00
79,2014210N10323,BERTHA,2014-08-02 09:00:00+00:00,2014-08-03 00:00:00+00:00


In [4]:
h3_cells =gpd .read_feather (weather_db_directory /'h3_cells_cached.feather')[['index','geometry']].copy ()

h3_cells_projected =h3_cells .to_crs ('EPSG:6933')
centroids =h3_cells_projected .geometry .centroid
cell_x =centroids .x .to_numpy (dtype =np .float32 )
cell_y =centroids .y .to_numpy (dtype =np .float32 )

print ('Number of cells:',h3_cells .shape [0 ])


Number of cells: 3571572


In [5]:
def to_utc_naive (ts ):
    ts =pd .Timestamp (ts )
    if ts .tzinfo is None :
        return ts
    return ts .tz_convert ('UTC').tz_localize (None )

def hourly_dates_for_storm (row ):
    storm_start =to_utc_naive (row ['datetime_min']).floor ('h')
    storm_end =to_utc_naive (row ['datetime_max']).ceil ('h')+pd .to_timedelta (1 ,'d')
    dates =pd .date_range (storm_start ,storm_end ,freq ='h')
    return dates ,storm_start ,storm_end

def interpolated_track_point (line_geom ,dt ,start_dt ,end_dt ):
    total_seconds =max ((end_dt -start_dt ).total_seconds (),1.0 )
    frac =(dt -start_dt ).total_seconds ()/total_seconds
    frac =float (np .clip (frac ,0.0 ,1.0 ))
    return line_geom .interpolate (frac ,normalized =True )


In [6]:
storms_projected =relevant_storms .to_crs ('EPSG:6933')

for _ ,row in tqdm (storms_projected .iterrows (),total =storms_projected .shape [0 ]):
    storm_id =row ['SID']
    storm_name =row ['NAME']

    dates ,storm_start ,storm_end =hourly_dates_for_storm (row )

    dates_df =pd .DataFrame ({'datetime':dates })
    dates_df ['file_name']=dates_df ['datetime'].dt .strftime ('%Y_%m_%dT%H_%M_%S')+'.npy'
    dates_df ['all_files_exist']=dates_df ['file_name'].apply (
    lambda f :all ((directory /f ).exists ()for directory in storm_track_mask_directories .values ())
    )

    missing_dates =dates_df [~dates_df ['all_files_exist']].reset_index (drop =True )

    if missing_dates .shape [0 ]==0 :
        continue

    storm_line =row .geometry

    for _ ,row_ in tqdm (missing_dates .iterrows (),total =missing_dates .shape [0 ],leave =False ):
        dt =row_ ['datetime']
        file_name =row_ ['file_name']

        point =interpolated_track_point (storm_line ,dt ,storm_start ,storm_end )

        dx =cell_x -np .float32 (point .x )
        dy =cell_y -np .float32 (point .y )
        dist_sq =dx *dx +dy *dy

        radii_to_write =missing_track_mask_radii (storm_track_mask_directories ,file_name )
        if len (radii_to_write )==0 :
            continue

        save_track_mask_variants (
        dist_sq =dist_sq ,
        file_name =file_name ,
        mask_directories =storm_track_mask_directories ,
        distance_threshold_sq_by_km =distance_threshold_sq_by_km ,
        radii_km =radii_to_write ,
        )


100%|██████████| 81/81 [00:50<00:00,  1.62it/s]   


In [7]:
mask_file_counts =count_track_mask_files (storm_track_mask_directories )
for radius_km ,count in mask_file_counts .items ():
    print (f'Saved mask files ({radius_km} km):',count )

mask_files =sorted (storm_track_mask_directory .glob ('*.npy'))

if len (mask_files )>0 :
    sample =mx .load (str (mask_files [0 ]))
    print ('Sample file (200 km):',mask_files [0 ].name )
    print ('Sample shape:',sample .shape )
    print ('Unique values in sample:',np .unique (np .array (sample )))


Saved mask files: 7331
Sample file: 2014_06_28T18_00_00.npy
Sample shape: (3571572,)
Unique values in sample: [-1  0]
